# Papilledema Research — Dataset Exploration

Exploratory inspection of **Kim Fundus** and **Original OCT** datasets using the shared `src/` validation, preprocessing, and split modules (zero duplicated logic).


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_all_configs
from src.data import (
    load_split_df,
    apply_fundus_clahe,
    load_preprocessed_pil,
    FUNDUS_CLASSES,
    OCT_3CLASSES,
    OCT_SEVERITY_CLASSES,
)

cfg = load_all_configs()
f_splits = load_split_df(cfg, "fundus_3class")
o3_splits = load_split_df(cfg, "oct_3class")
osev_splits = load_split_df(cfg, "oct_severity")
print("Fundus split counts:", f_splits["split"].value_counts().to_dict())
print("OCT 3-class split counts:", o3_splits["split"].value_counts().to_dict())
print("OCT severity split counts:", osev_splits["split"].value_counts().to_dict())


## 1. Class & Split Distributions


In [ ]:
display(pd.crosstab(f_splits["label"], f_splits["split"], margins=True))
display(pd.crosstab(o3_splits["label"], o3_splits["split"], margins=True))
display(pd.crosstab(osev_splits["label"], osev_splits["split"], margins=True))


## 2. Preprocessing & CLAHE Comparison (Fundus) and Grayscale Handling (OCT)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for j, cls in enumerate(FUNDUS_CLASSES):
    p = f_splits[f_splits["label"] == cls]["image_path"].iloc[0]
    raw_im = load_preprocessed_pil(p, "fundus", image_size=224, use_clahe=False)
    clahe_im = apply_fundus_clahe(raw_im)
    axes[0, j].imshow(raw_im)
    axes[0, j].set_title(f"{cls} (RGB)")
    axes[0, j].axis("off")
    axes[1, j].imshow(clahe_im)
    axes[1, j].set_title(f"{cls} (CLAHE)")
    axes[1, j].axis("off")
plt.tight_layout()
plt.show()


## 3. Fundus Near-Duplicate Cluster Inspection


In [ ]:
dups_path = Path(cfg["paths"]["manifests_dir"]) / "fundus_duplicates.csv"
if dups_path.exists():
    dups_df = pd.read_csv(dups_path)
    print("Total duplicate / near-duplicate pairs:", len(dups_df))
    display(dups_df.head(15))
    multi = f_splits[f_splits["cluster_size"] > 1].sort_values(["cluster_id", "sample_id"])
    if not multi.empty:
        first_cid = multi["cluster_id"].iloc[0]
        sample_grp = multi[multi["cluster_id"] == first_cid]
        fig, axes = plt.subplots(1, len(sample_grp), figsize=(4 * len(sample_grp), 3.5))
        if len(sample_grp) == 1:
            axes = [axes]
        for ax, (_, r) in zip(axes, sample_grp.iterrows()):
            ax.imshow(Image.open(r["image_path"]).convert("RGB"))
            ax.set_title(f"{r['sample_id']}\n({r['split']})", fontsize=8)
            ax.axis("off")
        plt.suptitle(f"Cluster {first_cid} (kept together in split '{sample_grp['split'].iloc[0]}')")
        plt.tight_layout()
        plt.show()


## 4. OCT Inferred Sequence Exploration


In [ ]:
seq_summary = o3_splits.groupby(["label", "inferred_sequence_id", "split_unit_type"]).agg(
    slices=("sample_id", "count"),
    splits=("split", lambda s: sorted(s.unique().tolist())),
).reset_index()
display(seq_summary)

# Show consecutive B-scans from one inferred sequence
seq_id = o3_splits[o3_splits["label"] == "Papilledema"]["inferred_sequence_id"].iloc[0]
seq_slices = o3_splits[o3_splits["inferred_sequence_id"] == seq_id].sort_values("slice_index").head(4)
fig, axes = plt.subplots(1, len(seq_slices), figsize=(12, 3))
for ax, (_, r) in zip(axes, seq_slices.iterrows()):
    ax.imshow(load_preprocessed_pil(r["image_path"], "oct", image_size=224), cmap="gray")
    ax.set_title(f"slice {r['slice_index']}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Ordered B-Scans from {seq_id} (inferred sequence)")
plt.tight_layout()
plt.show()
